In [1]:
import gymnasium as gym
import pygame
import numpy as np
import pickle

# Initialize environment and Pygame
env = gym.make("CartPole-v1", render_mode="rgb_array")
obs, _ = env.reset()

pygame.init()
screen = pygame.display.set_mode((600, 400))
pygame.display.set_caption("CartPole Manual Controller")

clock = pygame.time.Clock()
running = True

# Data storage
obs_list = []
action_list = []

def get_user_action():
    keys = pygame.key.get_pressed()
    if keys[pygame.K_LEFT]:
        return 0  # move left
    elif keys[pygame.K_RIGHT]:
        return 1  # move right
    else:
        return None  # no action

while running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    user_action = get_user_action()
    if user_action is not None:
        # Save state and action
        obs_list.append(obs)
        action_list.append(user_action)

        # Step in the environment
        obs, reward, terminated, truncated, _ = env.step(user_action)
        done = terminated or truncated
        if done:
            obs, _ = env.reset()

    # Render frame in pygame window
    frame = env.render()
    pygame.surfarray.blit_array(screen, np.transpose(frame, (1, 0, 2)))
    pygame.display.flip()
    clock.tick(30)

pygame.quit()
env.close()

# Save data
with open("cartpole_manual_demos.pkl", "wb") as f:
    pickle.dump((obs_list, action_list), f)

print("🎉 Saved", len(obs_list), "frames of expert data!")


🎉 Saved 0 frames of expert data!


In [1]:
import pickle

with open("cartpole_manual_demos.pkl", "rb") as f:
    obs_list, action_list = pickle.load(f)

print("Number of steps:", len(obs_list))
print("First observation:", obs_list[0])
print("First action:", action_list[0])


Number of steps: 1463
First observation: [-0.04514737 -0.00101127  0.00852235 -0.01635175]
First action: 1


In [2]:
import numpy as np
import pickle

with open("cartpole_manual_demos.pkl", "rb") as f:
    obs_list, action_list = pickle.load(f)

actions = np.array(action_list)
print("Action 0 count:", np.sum(actions == 0))
print("Action 1 count:", np.sum(actions == 1))


Action 0 count: 738
Action 1 count: 725


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pickle

# Load expert data
with open("cartpole_manual_demos.pkl", "rb") as f:
    obs_list, action_list = pickle.load(f)

# Convert to tensors
obs_tensor = torch.FloatTensor(obs_list)
actions_tensor = torch.LongTensor(action_list)

# Create DataLoader for batching
dataset = TensorDataset(obs_tensor, actions_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

# Define BC model
class BCModel(nn.Module):
    def __init__(self, input_dim=4, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

# Instantiate model, optimizer, loss function
bc_model = BCModel()
optimizer = optim.Adam(bc_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Training loop for BC
for epoch in range(50):
    for batch_obs, batch_act in dataloader:
        logits = bc_model(batch_obs)
        loss = criterion(logits, batch_act)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch + 1} | Loss: {loss.item():.4f}")

# Save BC model (after training)
torch.save(bc_model.state_dict(), "bc_model.pth")


C:\Users\Kaan\AppData\Local\Temp\ipykernel_15940\1815965741.py:13: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  obs_tensor = torch.FloatTensor(obs_list)


Epoch 1 | Loss: 0.6882
Epoch 2 | Loss: 0.6778
Epoch 3 | Loss: 0.6697
Epoch 4 | Loss: 0.6619
Epoch 5 | Loss: 0.6882
Epoch 6 | Loss: 0.6418
Epoch 7 | Loss: 0.6677
Epoch 8 | Loss: 0.6056
Epoch 9 | Loss: 0.6928
Epoch 10 | Loss: 0.6319
Epoch 11 | Loss: 0.6454
Epoch 12 | Loss: 0.6039
Epoch 13 | Loss: 0.6205
Epoch 14 | Loss: 0.6595
Epoch 15 | Loss: 0.6311
Epoch 16 | Loss: 0.6493
Epoch 17 | Loss: 0.5871
Epoch 18 | Loss: 0.5890
Epoch 19 | Loss: 0.5727
Epoch 20 | Loss: 0.6590
Epoch 21 | Loss: 0.5649
Epoch 22 | Loss: 0.5768
Epoch 23 | Loss: 0.5570
Epoch 24 | Loss: 0.5832
Epoch 25 | Loss: 0.6122
Epoch 26 | Loss: 0.5915
Epoch 27 | Loss: 0.6073
Epoch 28 | Loss: 0.6603
Epoch 29 | Loss: 0.6057
Epoch 30 | Loss: 0.6335
Epoch 31 | Loss: 0.5800
Epoch 32 | Loss: 0.4966
Epoch 33 | Loss: 0.5702
Epoch 34 | Loss: 0.5413
Epoch 35 | Loss: 0.6277
Epoch 36 | Loss: 0.6458
Epoch 37 | Loss: 0.5288
Epoch 38 | Loss: 0.6251
Epoch 39 | Loss: 0.5705
Epoch 40 | Loss: 0.5511
Epoch 41 | Loss: 0.5333
Epoch 42 | Loss: 0.5696
E

In [6]:
from stable_baselines3 import PPO
import gymnasium as gym
import torch
import torch.nn as nn

# Create CartPole environment
env = gym.make("CartPole-v1")

# Create PPO model (initially random weights)
model = PPO("MlpPolicy", env, verbose=1)

# Define BC model (same architecture as used for BC)
class BCModel(nn.Module):
    def __init__(self, input_dim=4, output_dim=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),  # Input to hidden layer
            nn.ReLU(),
            nn.Linear(64, output_dim)  # Hidden to output layer
        )
    
    def forward(self, x):
        return self.net(x)

# Load the trained BC model
bc_model = BCModel()
bc_model.load_state_dict(torch.load("bc_model.pth"))  # Load your saved BC model

# Access PPO policy network (action prediction part)
policy_net = model.policy.mlp_extractor.policy_net

# Check the dimensions of the layers
print("BC model layer shapes:", bc_model.net[0].weight.shape, bc_model.net[2].weight.shape)
print("PPO policy network layer shapes:", policy_net[0].weight.shape, policy_net[2].weight.shape)

# Transfer weights from BC model to PPO policy
with torch.no_grad():
    # Ensure we are matching layers
    if bc_model.net[0].weight.shape == policy_net[0].weight.shape:
        policy_net[0].weight.data.copy_(bc_model.net[0].weight.data)  # First Linear Layer weights
        policy_net[0].bias.data.copy_(bc_model.net[0].bias.data)  # First Linear Layer biases
    else:
        print("Mismatch in first layer dimensions!")
    
    if bc_model.net[2].weight.shape == policy_net[2].weight.shape:
        policy_net[2].weight.data.copy_(bc_model.net[2].weight.data)  # Second Linear Layer weights
        policy_net[2].bias.data.copy_(bc_model.net[2].bias.data)  # Second Linear Layer biases
    else:
        print("Mismatch in second layer dimensions!")

# Now that the BC model is transferred, let's continue RL fine-tuning
model.learn(total_timesteps=10000)

# Optionally, save the RL-finetuned model after training
model.save("ppo_cartpole_rl_finetuned")


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
BC model layer shapes: torch.Size([64, 4]) torch.Size([2, 64])
PPO policy network layer shapes: torch.Size([64, 4]) torch.Size([64, 64])
Mismatch in second layer dimensions!


C:\Users\Kaan\AppData\Local\Temp\ipykernel_15940\1228966107.py:27: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  bc_model.load_state_dict(torch.load("bc_model.pth"))  # Load

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.1     |
|    ep_rew_mean     | 21.1     |
| time/              |          |
|    fps             | 1583     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 29.6        |
|    ep_rew_mean          | 29.6        |
| time/                   |             |
|    fps                  | 1136        |
|    iterations           | 2           |
|    time_elapsed         | 3           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009477691 |
|    clip_fraction        | 0.12        |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | -0.00366    |
|    learning_rate        | 0.